In [1]:
from test.helpers import sde_solver_order

import diffrax
import equinox as eqx
import jax.numpy as jnp
import jax.random as jrandom
import jax.tree_util as jtu
from jax import config


config.update("jax_enable_x64", True)


def _squareplus(x):
    return 0.5 * (x + jnp.sqrt(x**2 + 4))


def sde_strong_order(solver_ctr, commutative, theoretical_order):
    key = jrandom.PRNGKey(5678)
    driftkey, diffusionkey, ykey, bmkey = jrandom.split(key, 4)
    num_samples = 100
    bmkeys = jrandom.split(bmkey, num=num_samples)

    if commutative:
        noise_dim = 1
    else:
        noise_dim = 5

    def drift(t, y, args):
        mlp = eqx.nn.MLP(
            in_size=3,
            out_size=3,
            width_size=8,
            depth=1,
            activation=_squareplus,
            key=driftkey,
        )
        return 0.5 * mlp(y)

    def diffusion(t, y, args):
        mlp = eqx.nn.MLP(
            in_size=3,
            out_size=3 * noise_dim,
            width_size=8,
            depth=1,
            activation=_squareplus,
            final_activation=jnp.tanh,
            key=diffusionkey,
        )
        return 0.25 * mlp(y).reshape(3, noise_dim)

    t0 = 0
    t1 = 2
    y0 = jrandom.normal(ykey, (3,), dtype=jnp.float64)

    sde = (drift, diffusion, None, y0, t0, t1, noise_dim)

    # Reference solver is always an ODE-viable solver, so its implementation has been
    # verified by the ODE tests like test_ode_order.
    if issubclass(solver_ctr, diffrax.AbstractItoSolver):
        ref_solver = diffrax.Euler()
    elif issubclass(solver_ctr, diffrax.AbstractStratonovichSolver):
        ref_solver = diffrax.Heun()
    else:
        assert False

    hs, errors, order = sde_solver_order(
        bmkeys, sde, solver_ctr(), ref_solver, 2**-14, hs_num=7
    )
    log_errs = jnp.log2(errors)
    slopes = log_errs[:-1] - log_errs[1:]
    assert -0.2 < order - theoretical_order < 0.2
    return order, slopes

In [2]:
def _solvers():
    # solver, commutative, order
    yield diffrax.Euler, False, "Euler"  # PASSES
    yield diffrax.EulerHeun, False, "EulerHeun"  # 0.9326
    yield diffrax.Heun, False, "Heun"  # 0.86656
    yield diffrax.ItoMilstein, False, "ItoMilstein"  # 1.0025
    yield diffrax.Midpoint, False, "Midpoint"  # 0.8659
    yield diffrax.ReversibleHeun, False, "ReversibleHeun"  # 0.8666
    yield diffrax.StratonovichMilstein, False, "StratonovichMilstein"  # 0.9331
    yield diffrax.ReversibleHeun, True, "ReversibleHeun"  # 1.3648
    yield diffrax.StratonovichMilstein, True, "StratonovichMilstein"  # PASSES

In [3]:
slopes_dict = {}
for solver, comm, name in _solvers():
    order, slopes = sde_strong_order(solver, comm)
    slopes_dict.update({name: slopes})
    pretty_slopes = ["{0:0.5f}".format(slope) for slope in slopes]
    print(f"{name} has slopes {pretty_slopes} and order {order:.5f}")

Euler has slopes ['0.80191', '0.66703', '0.32430', '0.53782', '0.44805', '0.37029'] and order 0.50945
EulerHeun has slopes ['0.71669', '0.87360', '0.29220', '0.56652', '0.49204', '0.68681'] and order 0.57825
Heun has slopes ['0.57572', '0.85757', '0.13873', '0.54256', '0.47378', '0.66119'] and order 0.51626
ItoMilstein has slopes ['0.71535', '0.87469', '0.30138', '0.55191', '0.47130', '0.66119'] and order 0.57069
Midpoint has slopes ['0.57397', '0.86080', '0.13611', '0.54239', '0.47405', '0.66096'] and order 0.51607
ReversibleHeun has slopes ['0.57591', '0.85719', '0.13902', '0.54254', '0.47369', '0.66119'] and order 0.51625
StratonovichMilstein has slopes ['0.71600', '0.87569', '0.29244', '0.56685', '0.49206', '0.68670'] and order 0.57866
ReversibleHeun has slopes ['0.75643', '0.96169', '0.91476', '1.13479', '0.78509', '0.92417'] and order 0.93118
StratonovichMilstein has slopes ['1.01076', '0.99114', '1.00239', '0.99731', '1.01628', '0.97563'] and order 0.99980


In [4]:
pretty_slopes_dict = jtu.tree_map(lambda x: "{0:0.5f}".format(x), slopes_dict)
for key, value in pretty_slopes_dict.items():
    print(f"{key}:   {value}")

TypeError: unsupported format string passed to numpy.ndarray.__format__